In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# Data
BATCH_SIZE = 128
NUM_WORKERS = 4
VAL_FRACTION = 0.1

In [3]:
def get_cifar10_loaders(batch_size=128, val_fraction=0.1, num_workers=4, seed=0):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_full = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

    n_total = len(train_full)
    n_val = int(val_fraction * n_total)
    n_train = n_total - n_val

    g = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(train_full, [n_train, n_val], generator=g)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [4]:
def get_cifar10_bn_loader(batch_size=128):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_set = torchvision.datasets.CIFAR10("./data", train=True, download=False, transform=tf)
    return DataLoader(train_set, batch_size=batch_size, shuffle=False)

bn_loader = get_cifar10_bn_loader(
    batch_size=BATCH_SIZE
)


In [5]:
class ChannelGate(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Parameter(torch.ones(channels))

    def forward(self, x):
        return x * self.gate.view(1, -1, 1, 1)


class WideBasic(nn.Module):
    def __init__(self, in_planes, planes, dropout_rate, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, padding=1, bias=False)
        self.gate1 = ChannelGate(planes)

        self.dropout = nn.Dropout(p=dropout_rate) if dropout_rate > 0 else nn.Identity()

        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.gate2 = ChannelGate(planes)

        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.gate1(out)
        out = self.dropout(out)

        out = self.conv2(F.relu(self.bn2(out)))
        out = self.gate2(out)

        out += self.shortcut(x)
        return out


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=10, dropout_rate=0.0, num_classes=100):
        super().__init__()
        assert (depth - 4) % 6 == 0, "WRN depth must be 6n+4"
        n = (depth - 4) // 6
        k = widen_factor

        stages = [16, 16*k, 32*k, 64*k]

        self.conv1 = nn.Conv2d(3, stages[0], 3, padding=1, bias=False)
        self.layer1 = self._make_layer(stages[0], stages[1], n, dropout_rate, stride=1)
        self.layer2 = self._make_layer(stages[1], stages[2], n, dropout_rate, stride=2)
        self.layer3 = self._make_layer(stages[2], stages[3], n, dropout_rate, stride=2)
        self.bn1 = nn.BatchNorm2d(stages[3])
        self.fc = nn.Linear(stages[3], num_classes)

        self._init_weights()

    def _make_layer(self, in_planes, planes, num_blocks, dropout_rate, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        curr_in = in_planes
        for s in strides:
            layers.append(WideBasic(curr_in, planes, dropout_rate, stride=s))
            curr_in = planes
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1).view(out.size(0), -1)
        return self.fc(out)


def build_wrn28_10(num_classes=10, dropout=0.0):
    return WideResNet(depth=28, widen_factor=10, dropout_rate=dropout, num_classes=num_classes)


# sanity check
m = build_wrn28_10().to(DEVICE)
print(m(torch.randn(2, 3, 32, 32).to(DEVICE)).shape)  # [2, 100]


torch.Size([2, 10])


In [6]:
import os

def gate_l1_loss(model):
    loss = 0.0
    for m in model.modules():
        if isinstance(m, ChannelGate):
            loss += torch.sum(torch.abs(m.gate))
    return loss


@torch.no_grad()
def evaluate_acc(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = model(x).argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return 100 * correct / total


def train_gated(model, train_loader, val_loader,
                epochs=200, lr=0.1, weight_decay=5e-4,
                sparse_lambda=1e-4,
                save_dir="Baseline_Gate_Decorator/WRN28-10_CIFAR10"):

    os.makedirs(save_dir, exist_ok=True)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, epochs
    )

    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = ce_loss(logits, y)
            loss += sparse_lambda * gate_l1_loss(model)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        val_acc = evaluate_acc(model, val_loader)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Loss {total_loss/len(train_loader):.4f} | "
              f"Val Acc {val_acc:.2f}")

    # Save final model only (after full training)
    torch.save(
        model.state_dict(),
        os.path.join(save_dir, "final_model.pth")
    )

    print("Training finished. Final model saved.")

In [18]:
train_loader, val_loader, test_loader = get_cifar10_loaders()

In [ ]:
train_loader, val_loader, test_loader = get_cifar10_loaders()

model = build_wrn28_10(num_classes=10).to(DEVICE)

train_gated(model, train_loader, val_loader)

In [7]:
@torch.no_grad()
def wrn_stage_gate_scores(model):
    def stage_scores(layer):
        per_block = []
        for blk in layer:
            g1 = blk.gate1.gate.abs()
            g2 = blk.gate2.gate.abs()
            per_block.append((g1 + g2) / 2)
        return torch.stack(per_block).mean(0)

    return {
        "layer1": stage_scores(model.layer1),
        "layer2": stage_scores(model.layer2),
        "layer3": stage_scores(model.layer3),
    }

In [8]:
def select_indices(scores, keep_ratio=0.5, min_channels=8):
    C = scores.numel()
    k = max(min_channels, int(C * keep_ratio))
    idx = torch.topk(scores, k).indices
    return torch.sort(idx).values

In [81]:
model = build_wrn28_10(num_classes=10).to(DEVICE)
model.load_state_dict(torch.load(
    "Baseline_Gate_Decorator/WRN28-10_CIFAR10/final_model.pth"
))
model.eval()

scores = wrn_stage_gate_scores(model)

print("Layer1 channels:", scores["layer1"].shape)
print("Layer2 channels:", scores["layer2"].shape)
print("Layer3 channels:", scores["layer3"].shape)

Layer1 channels: torch.Size([160])
Layer2 channels: torch.Size([320])
Layer3 channels: torch.Size([640])


In [82]:
keep_ratio = 0.1

k1 = select_indices(scores["layer1"], keep_ratio)
k2 = select_indices(scores["layer2"], keep_ratio)
k3 = select_indices(scores["layer3"], keep_ratio)

print("Keep L1:", len(k1))
print("Keep L2:", len(k2))
print("Keep L3:", len(k3))

Keep L1: 16
Keep L2: 32
Keep L3: 64


In [83]:
class WideBasicNoGate(nn.Module):
    def __init__(self, in_planes, planes, dropout_rate, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, padding=1, bias=False)
        self.dropout = nn.Dropout(p=dropout_rate) if dropout_rate > 0 else nn.Identity()
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)

        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.dropout(out)
        out = self.conv2(F.relu(self.bn2(out)))
        out += self.shortcut(x)
        return out

In [84]:
class WideResNetNoGate(nn.Module):
    def __init__(self, stages, depth=28, dropout_rate=0.0, num_classes=10):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6

        self.conv1 = nn.Conv2d(3, stages[0], 3, padding=1, bias=False)
        self.layer1 = self._make_layer(stages[0], stages[1], n, dropout_rate, 1)
        self.layer2 = self._make_layer(stages[1], stages[2], n, dropout_rate, 2)
        self.layer3 = self._make_layer(stages[2], stages[3], n, dropout_rate, 2)
        self.bn1 = nn.BatchNorm2d(stages[3])
        self.fc = nn.Linear(stages[3], num_classes)

    def _make_layer(self, in_planes, planes, num_blocks, dropout_rate, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        curr = in_planes
        for s in strides:
            layers.append(WideBasicNoGate(curr, planes, dropout_rate, s))
            curr = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1).view(out.size(0), -1)
        return self.fc(out)

In [85]:
@torch.no_grad()
def melt_wrn(model, k1, k2, k3, num_classes=10):

    C0 = model.conv1.out_channels
    C1 = len(k1)
    C2 = len(k2)
    C3 = len(k3)

    pruned = WideResNetNoGate(
        stages=[C0, C1, C2, C3],
        depth=28,
        num_classes=num_classes
    ).to(DEVICE)

    # Copy stem
    pruned.conv1.weight.data.copy_(model.conv1.weight.data)

    def copy_stage(stage_new, stage_old, in_idx, out_idx):
        for i, (bnew, bold) in enumerate(zip(stage_new, stage_old)):

            if i == 0 and in_idx is None:
                bn1_idx = torch.arange(model.conv1.out_channels).to(DEVICE)
            elif i == 0:
                bn1_idx = in_idx
            else:
                bn1_idx = out_idx

            # BN1
            bnew.bn1.weight.data.copy_(bold.bn1.weight.data[bn1_idx])
            bnew.bn1.bias.data.copy_(bold.bn1.bias.data[bn1_idx])
            bnew.bn1.running_mean.data.copy_(bold.bn1.running_mean.data[bn1_idx])
            bnew.bn1.running_var.data.copy_(bold.bn1.running_var.data[bn1_idx])

            # Conv1
            w = bold.conv1.weight.data[out_idx][:, bn1_idx]
            bnew.conv1.weight.data.copy_(w)

            # BN2
            bnew.bn2.weight.data.copy_(bold.bn2.weight.data[out_idx])
            bnew.bn2.bias.data.copy_(bold.bn2.bias.data[out_idx])
            bnew.bn2.running_mean.data.copy_(bold.bn2.running_mean.data[out_idx])
            bnew.bn2.running_var.data.copy_(bold.bn2.running_var.data[out_idx])

            # Conv2
            w2 = bold.conv2.weight.data[out_idx][:, out_idx]
            bnew.conv2.weight.data.copy_(w2)

            # Shortcut
            if not isinstance(bold.shortcut, nn.Identity):
                w_sc = bold.shortcut.weight.data[out_idx][:, bn1_idx]
                bnew.shortcut.weight.data.copy_(w_sc)

    copy_stage(pruned.layer1, model.layer1, None, k1)
    copy_stage(pruned.layer2, model.layer2, k1, k2)
    copy_stage(pruned.layer3, model.layer3, k2, k3)

    # Final BN + FC
    pruned.bn1.weight.data.copy_(model.bn1.weight.data[k3])
    pruned.bn1.bias.data.copy_(model.bn1.bias.data[k3])
    pruned.bn1.running_mean.data.copy_(model.bn1.running_mean.data[k3])
    pruned.bn1.running_var.data.copy_(model.bn1.running_var.data[k3])

    pruned.fc.weight.data.copy_(model.fc.weight.data[:, k3])
    pruned.fc.bias.data.copy_(model.fc.bias.data)

    return pruned

In [87]:
## for 90% pruning 

@torch.no_grad()
def melt_wrn(model, k1, k2, k3, num_classes=10):

    C0 = model.conv1.out_channels
    C1 = len(k1)
    C2 = len(k2)
    C3 = len(k3)

    pruned = WideResNetNoGate(
        stages=[C0, C1, C2, C3],
        depth=28,
        num_classes=num_classes
    ).to(DEVICE)

    # ---------------- Stem ----------------
    pruned.conv1.weight.data.copy_(model.conv1.weight.data)

    def copy_stage(stage_new, stage_old, in_idx, out_idx):
        for i, (bnew, bold) in enumerate(zip(stage_new, stage_old)):

            if i == 0 and in_idx is None:
                bn1_idx = torch.arange(model.conv1.out_channels).to(DEVICE)
            elif i == 0:
                bn1_idx = in_idx
            else:
                bn1_idx = out_idx

            # ---------- BN1 ----------
            bnew.bn1.weight.data.copy_(bold.bn1.weight.data[bn1_idx])
            bnew.bn1.bias.data.copy_(bold.bn1.bias.data[bn1_idx])
            bnew.bn1.running_mean.data.copy_(bold.bn1.running_mean.data[bn1_idx])
            bnew.bn1.running_var.data.copy_(bold.bn1.running_var.data[bn1_idx])

            # ---------- Conv1 ----------
            w1 = bold.conv1.weight.data[out_idx][:, bn1_idx]
            bnew.conv1.weight.data.copy_(w1)

            # ---------- BN2 ----------
            bnew.bn2.weight.data.copy_(bold.bn2.weight.data[out_idx])
            bnew.bn2.bias.data.copy_(bold.bn2.bias.data[out_idx])
            bnew.bn2.running_mean.data.copy_(bold.bn2.running_mean.data[out_idx])
            bnew.bn2.running_var.data.copy_(bold.bn2.running_var.data[out_idx])

            # ---------- Conv2 ----------
            w2 = bold.conv2.weight.data[out_idx][:, out_idx]
            bnew.conv2.weight.data.copy_(w2)

            # ---------- Shortcut (FIXED) ----------
            if not isinstance(bold.shortcut, nn.Identity):
                if not isinstance(bnew.shortcut, nn.Identity):
                    w_sc = bold.shortcut.weight.data[out_idx][:, bn1_idx]
                    bnew.shortcut.weight.data.copy_(w_sc)
                # else:
                # new block does not require projection shortcut
                # nothing to copy

    # Copy stages
    copy_stage(pruned.layer1, model.layer1, None, k1)
    copy_stage(pruned.layer2, model.layer2, k1, k2)
    copy_stage(pruned.layer3, model.layer3, k2, k3)

    # ---------------- Final BN ----------------
    pruned.bn1.weight.data.copy_(model.bn1.weight.data[k3])
    pruned.bn1.bias.data.copy_(model.bn1.bias.data[k3])
    pruned.bn1.running_mean.data.copy_(model.bn1.running_mean.data[k3])
    pruned.bn1.running_var.data.copy_(model.bn1.running_var.data[k3])

    # ---------------- FC ----------------
    pruned.fc.weight.data.copy_(model.fc.weight.data[:, k3])
    pruned.fc.bias.data.copy_(model.fc.bias.data)

    return pruned

In [88]:
pruned_model_90 = melt_wrn(model, k1, k2, k3)

In [89]:
@torch.no_grad()
def recalibrate_bn(model, loader):
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.running_mean.zero_()
            m.running_var.fill_(1)

    for x, _ in loader:
        x = x.to(DEVICE)
        model(x)

    model.eval()

In [90]:
recalibrate_bn(pruned_model_90, bn_loader)

In [68]:
acc_post_melt = evaluate_acc(pruned_model_80, test_loader)
print("Accuracy after melt (before finetune):", acc_post_melt)

Accuracy after melt (before finetune): 22.58


In [91]:
def finetune(model, train_loader, val_loader,
             epochs=40, lr=0.01):

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, epochs
    )

    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = ce_loss(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()

        # validation accuracy
        val_acc = evaluate_acc(model, val_loader)

        print(f"[FineTune] Epoch {epoch+1}/{epochs} | "
              f"Loss {running_loss/len(train_loader):.4f} | "
              f"Val Acc {val_acc:.2f}")

In [92]:
import os

ft_dir = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/FINE_TUNED"
os.makedirs(ft_dir, exist_ok=True)

pruned_model = pruned_model_90.to(DEVICE)

finetune(
    model=pruned_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=40,
    lr=0.01
)

torch.save(
    pruned_model.state_dict(),
    os.path.join(ft_dir, "wrn28_10_pruned_90_finetuned.pth")
)

[FineTune] Epoch 1/40 | Loss 1.8366 | Val Acc 29.96
[FineTune] Epoch 2/40 | Loss 1.3279 | Val Acc 55.80
[FineTune] Epoch 3/40 | Loss 1.0352 | Val Acc 59.40
[FineTune] Epoch 4/40 | Loss 0.8918 | Val Acc 64.34
[FineTune] Epoch 5/40 | Loss 0.7930 | Val Acc 72.46
[FineTune] Epoch 6/40 | Loss 0.6966 | Val Acc 73.98
[FineTune] Epoch 7/40 | Loss 0.6273 | Val Acc 64.62
[FineTune] Epoch 8/40 | Loss 0.5812 | Val Acc 74.36
[FineTune] Epoch 9/40 | Loss 0.5441 | Val Acc 77.30
[FineTune] Epoch 10/40 | Loss 0.5047 | Val Acc 78.76
[FineTune] Epoch 11/40 | Loss 0.4803 | Val Acc 78.92
[FineTune] Epoch 12/40 | Loss 0.4596 | Val Acc 78.08
[FineTune] Epoch 13/40 | Loss 0.4358 | Val Acc 82.20
[FineTune] Epoch 14/40 | Loss 0.4198 | Val Acc 81.44
[FineTune] Epoch 15/40 | Loss 0.4012 | Val Acc 83.86
[FineTune] Epoch 16/40 | Loss 0.3803 | Val Acc 84.66
[FineTune] Epoch 17/40 | Loss 0.3656 | Val Acc 84.12
[FineTune] Epoch 18/40 | Loss 0.3497 | Val Acc 84.64
[FineTune] Epoch 19/40 | Loss 0.3343 | Val Acc 84.38
[F

In [93]:
@torch.no_grad()
def collect_logits_and_labels(model, loader, device):
    model.eval()
    logits_list, labels_list = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        logits_list.append(logits.detach().cpu())
        labels_list.append(y.detach().cpu())
    return torch.cat(logits_list, 0), torch.cat(labels_list, 0)

@torch.no_grad()
def acc_from_logits(logits, labels):
    return float((logits.argmax(1) == labels).float().mean().item())

@torch.no_grad()
def nll_from_logits(logits, labels):
    return float(F.cross_entropy(logits, labels, reduction="mean").item())

@torch.no_grad()
def ece_from_logits(logits, labels, n_bins=15, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    conf, pred = probs.max(1)
    acc = pred.eq(labels).float()

    edges = torch.linspace(0, 1, n_bins + 1, device=logits.device)
    ece = torch.zeros(1, device=logits.device)

    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        inb = (conf > lo) & (conf <= hi)
        p = inb.float().mean()
        if p.item() > 0:
            ece += p * torch.abs(conf[inb].mean() - acc[inb].mean())
    return float(ece.item())

@torch.no_grad()
def ks_calibration_error(logits, labels, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    conf, pred = probs.max(1)
    correct = pred.eq(labels).float()

    conf_sorted, idx = torch.sort(conf)
    corr_sorted = correct[idx]
    denom = torch.arange(1, len(conf_sorted)+1, device=logits.device).float()
    cum_acc = torch.cumsum(corr_sorted, 0) / denom
    cum_conf = torch.cumsum(conf_sorted, 0) / denom
    return float(torch.max(torch.abs(cum_acc - cum_conf)).item())

@torch.no_grad()
def mean_sweep_ce(logits, labels, bin_list, temperature=1.0):
    vals = [ece_from_logits(logits, labels, n_bins=b, temperature=temperature) for b in bin_list]
    return float(np.mean(vals))

def fit_temperature(logits_val_cpu, labels_val_cpu, device, max_iter=50):
    logits = logits_val_cpu.to(device)
    labels = labels_val_cpu.to(device)

    logT = torch.zeros(1, device=device, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.5, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        T = torch.exp(logT)
        loss = F.cross_entropy(logits / T, labels)
        loss.backward()
        return loss

    opt.step(closure)
    T = float(torch.exp(logT).detach().cpu().item())
    return max(1e-3, min(T, 100.0))

@torch.no_grad()
def evaluate_all_metrics(model, val_loader, test_loader, device):
    logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
    logits_test, labels_test = collect_logits_and_labels(model, test_loader, device)

    logits_test = logits_test.to(device)
    labels_test = labels_test.to(device)

    # base
    acc = acc_from_logits(logits_test, labels_test)
    nll = nll_from_logits(logits_test, labels_test)
    ece = ece_from_logits(logits_test, labels_test, n_bins=ECE_BINS)
    ks  = ks_calibration_error(logits_test, labels_test)
    ms  = mean_sweep_ce(logits_test, labels_test, MEAN_SWEEP_BINS)

    # TS
    T = fit_temperature(logits_val, labels_val, device=device)
    ts_nll = nll_from_logits(logits_test / T, labels_test)
    ts_ece = ece_from_logits(logits_test, labels_test, n_bins=ECE_BINS, temperature=T)
    ts_ks  = ks_calibration_error(logits_test, labels_test, temperature=T)
    ts_ms  = mean_sweep_ce(logits_test, labels_test, MEAN_SWEEP_BINS, temperature=T)

    return {
        "acc": acc, "nll": nll, "ece": ece, "ts_ece": ts_ece,
        "ks": ks, "ts_ks": ts_ks,
        "mean_sweep_ce": ms, "ts_mean_sweep_ce": ts_ms,
        "ts_temperature": T,
        "ts_nll": ts_nll,
    }


In [102]:
import os
import pandas as pd
import torch

# -------------------------------------------------
# Build pruned WRN architecture from prune ratio
# -------------------------------------------------
def build_pruned_wrn_from_ratio(prune_ratio, num_classes=10):
    keep_ratio = 1 - prune_ratio

    C0 = 16
    C1 = int(round(160 * keep_ratio))
    C2 = int(round(320 * keep_ratio))
    C3 = int(round(640 * keep_ratio))

    return WideResNetNoGate(
        stages=[C0, C1, C2, C3],
        depth=28,
        num_classes=num_classes
    )


# -------------------------------------------------
# Evaluate single checkpoint
# -------------------------------------------------
def evaluate_checkpoint(ckpt_path, prune_ratio, val_loader, test_loader, device):
    
    if prune_ratio == 0.0:
        model = build_wrn28_10(num_classes=10)
    else:
        model = build_pruned_wrn_from_ratio(prune_ratio, num_classes=10)

    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)

    model.to(device)
    model.eval()

    metrics = evaluate_all_metrics(
        model=model,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device
    )

    return metrics


# -------------------------------------------------
# Evaluate baseline + pruned sweep
# -------------------------------------------------
def evaluate_gate_decorator_models(
    baseline_ckpt,
    sweep_dir,
    prune_ratios,
    device,
):

    _, val_loader, test_loader = get_cifar10_loaders()

    rows = []

    # -------- Baseline --------
    base_metrics = evaluate_checkpoint(
        baseline_ckpt,
        prune_ratio=0.0,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device
    )

    rows.append({
        "model": "baseline",
        "prune_ratio": 0.0,
        **base_metrics
    })

    # -------- Pruned --------
    for p in prune_ratios:

        tag = f"pruned_{int(p*100)}"
        ckpt_path = os.path.join(
            sweep_dir,
            f"wrn28_10_pruned_{int(p*100)}_finetuned.pth"
        )

        if not os.path.exists(ckpt_path):
            print(f"[WARN] Missing: {ckpt_path}")
            continue

        m = evaluate_checkpoint(
            ckpt_path,
            prune_ratio=p,
            val_loader=val_loader,
            test_loader=test_loader,
            device=device
        )

        rows.append({
            "model": tag,
            "prune_ratio": p,
            "keep_ratio": 1 - p,
            **m
        })

    return pd.DataFrame(rows)

In [104]:
# Metrics
ECE_BINS = 15
MEAN_SWEEP_BINS = list(range(5, 31))

BASELINE_CKPT = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/final_model.pth"

SWEEP_DIR = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/FINE_TUNED"

df = evaluate_gate_decorator_models(
    baseline_ckpt=BASELINE_CKPT,
    sweep_dir=SWEEP_DIR,
    prune_ratios=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],   # add more if available
    
    device=DEVICE,
)

df

,model,prune_ratio,acc,nll,ece,ts_ece,ks,ts_ks,mean_sweep_ce,ts_mean_sweep_ce,ts_temperature,ts_nll,keep_ratio
0,baseline,0.0,0.9506,0.187703,0.028074,0.014863,0.741059,0.778201,0.028232,0.014731,1.345361,0.164267,NaN
1,pruned_10,0.1,0.9555,0.177020,0.025458,0.010588,0.321566,0.309983,0.025478,0.010861,1.405951,0.151358,0.9
2,pruned_20,0.2,0.9533,0.184308,0.027535,0.013187,0.253971,0.235857,0.027518,0.013589,1.380612,0.157536,0.8
3,pruned_30,0.3,0.9541,0.180211,0.026477,0.011810,0.249786,0.339451,0.026479,0.011235,1.418385,0.153259,0.7
4,pruned_40,0.4,0.9503,0.192863,0.029823,0.015458,0.718825,0.767007,0.029874,0.015507,1.398372,0.162666,0.6
5,pruned_50,0.5,0.9517,0.187460,0.027411,0.012231,0.709865,0.249125,0.027738,0.012222,1.414630,0.159265,0.5
6,pruned_60,0.6,0.9471,0.204641,0.030468,0.014676,0.380032,0.314390,0.030468,0.014703,1.440280,0.171496,0.4
7,pruned_70,0.7,0.9338,0.239575,0.034285,0.012267,0.744505,0.786052,0.034357,0.012512,1.464905,0.204225,0.3
8,pruned_80,0.8,0.9148,0.295411,0.038908,0.012830,0.766879,0.781640,0.039000,0.014043,1.409389,0.260497,0.2
9,pruned_90,0.9,0.8927,0.334507,0.031165,0.013299,0.419482,0.437366,0.031309,0.013974,1.203531,0.320371,0.1


In [105]:
## Bootsrapping

In [122]:
def bootstrap_metrics_from_logits(
    logits_val_cpu,
    labels_val_cpu,
    logits_test_cpu,
    labels_test_cpu,
    device,
    n_boot=300,
):
    """
    Bootstrap uncertainty for:
    acc, nll, ece, ks, mean_sweep_ce,
    ts_ece, ts_nll, ts_ks, ts_mean_sweep_ce
    """

    N = logits_test_cpu.shape[0]
    results = []

    for _ in range(n_boot):

        # ---- Resample TEST set ----
        idx = torch.randint(0, N, (N,))
        logits_test_b = logits_test_cpu[idx].to(device)
        labels_test_b = labels_test_cpu[idx].to(device)

        # ---- Base metrics ----
        acc = acc_from_logits(logits_test_b, labels_test_b)
        nll = nll_from_logits(logits_test_b, labels_test_b)
        ece = ece_from_logits(logits_test_b, labels_test_b, n_bins=ECE_BINS)
        ks  = ks_calibration_error(logits_test_b, labels_test_b)
        ms  = mean_sweep_ce(logits_test_b, labels_test_b, MEAN_SWEEP_BINS)

        # ---- Temperature Scaling (fit on VAL each time) ----
        T = fit_temperature(
            logits_val_cpu,
            labels_val_cpu,
            device=device
        )

        ts_nll = nll_from_logits(logits_test_b / T, labels_test_b)
        ts_ece = ece_from_logits(logits_test_b, labels_test_b, n_bins=ECE_BINS, temperature=T)
        ts_ks  = ks_calibration_error(logits_test_b, labels_test_b, temperature=T)
        ts_ms  = mean_sweep_ce(logits_test_b, labels_test_b, MEAN_SWEEP_BINS, temperature=T)

        results.append({
            "acc": acc,
            "nll": nll,
            "ece": ece,
            "ks": ks,
            "mean_sweep_ce": ms,
            "ts_ece": ts_ece,
            "ts_nll": ts_nll,
            "ts_ks": ts_ks,
            "ts_mean_sweep_ce": ts_ms,
        })

    df_boot = pd.DataFrame(results)

    # Return only std
    return {k + "_std": float(df_boot[k].std()) for k in df_boot.columns}

In [129]:
@torch.no_grad()
def evaluate_checkpoint_with_bootstrap(
    ckpt_path: str,
    prune_ratio: float,
    val_loader,
    test_loader,
    device,
    n_boot=300,
):
    # -----------------------
    # Load checkpoint
    # -----------------------
    state_dict = torch.load(ckpt_path, map_location=device)

    # --------------------------------------------------
    # CASE 1: Baseline (prune_ratio == 0)
    # --------------------------------------------------
    if prune_ratio == 0.0:
        model = build_wrn28_10(
            num_classes=10,
            dropout=0.0
        ).to(device)

    # --------------------------------------------------
    # CASE 2: Structured pruned model (WRN-28-10)
    # --------------------------------------------------
    else:
        keep_ratio = 1.0 - prune_ratio

        BASE_C1 = 160
        BASE_C2 = 320
        BASE_C3 = 640

        C0 = 16
        C1 = max(1, int(BASE_C1 * keep_ratio))
        C2 = max(1, int(BASE_C2 * keep_ratio))
        C3 = max(1, int(BASE_C3 * keep_ratio))

        model = WideResNetNoGate(
            stages=[C0, C1, C2, C3],
            depth=28,
            num_classes=10
        ).to(device)

    model.load_state_dict(state_dict, strict=True)
    model.eval()

    # -----------------------
    # Collect logits once
    # -----------------------
    logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
    logits_test, labels_test = collect_logits_and_labels(model, test_loader, device)

    logits_val_cpu = logits_val.detach().cpu()
    labels_val_cpu = labels_val.detach().cpu()
    logits_test_cpu = logits_test.detach().cpu()
    labels_test_cpu = labels_test.detach().cpu()

    # -----------------------
    # Compute mean metrics
    # -----------------------
    metrics = evaluate_all_metrics(model, val_loader, test_loader, device)

    # -----------------------
    # Bootstrap std
    # -----------------------
    boot = bootstrap_metrics_from_logits(
        logits_val_cpu,
        labels_val_cpu,
        logits_test_cpu,
        labels_test_cpu,
        device,
        n_boot=n_boot
    )

    return {**metrics, **boot}

In [130]:
def evaluate_baseline_and_pruned_models(
    baseline_ckpt: str,
    sweep_dir: str,
    prune_ratios,
    device,
    n_boot=300,
):

    _, val_loader, test_loader = get_cifar10_loaders(
        batch_size=BATCH_SIZE,
        val_fraction=VAL_FRACTION,
        num_workers=NUM_WORKERS,
        seed=SEED,
    )

    rows = []

    # ------------------ BASELINE ------------------
    print("\nEvaluating BASELINE")

    base_metrics = evaluate_checkpoint_with_bootstrap(
        ckpt_path=baseline_ckpt,
        prune_ratio=0.0,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        n_boot=n_boot
    )

    rows.append({
        "model": "baseline",
        "prune_ratio": 0.0,
        "keep_ratio": 1.0,
        **base_metrics
    })

    # ------------------ PRUNED MODELS ------------------
    for p in prune_ratios:

        tag = f"wrn28_10_pruned_{int(p*100)}_finetuned"
        ckpt_path = os.path.join(sweep_dir, f"{tag}.pth")

        if not os.path.exists(ckpt_path):
            print(f"[WARN] Missing: {ckpt_path}")
            continue

        print(f"\nEvaluating {tag}")

        m = evaluate_checkpoint_with_bootstrap(
            ckpt_path=ckpt_path,
            prune_ratio=p,
            val_loader=val_loader,
            test_loader=test_loader,
            device=device,
            n_boot=n_boot
        )

        rows.append({
            "model": tag,
            "prune_ratio": p,
            "keep_ratio": 1.0 - p,
            **m
        })

    df = pd.DataFrame(rows)
    return df

In [133]:
SEED = 0
BASELINE_CKPT = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/final_model.pth"

SWEEP_DIR = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/FINE_TUNED"


df = evaluate_baseline_and_pruned_models(
    baseline_ckpt=BASELINE_CKPT,
    sweep_dir=SWEEP_DIR,
    prune_ratios=[0.1,0.2,0.3,0.4,0.5,0.6,0.7],
    device=DEVICE,
    n_boot=300   
)

df


Evaluating BASELINE

Evaluating wrn28_10_pruned_10_finetuned

Evaluating wrn28_10_pruned_20_finetuned

Evaluating wrn28_10_pruned_30_finetuned

Evaluating wrn28_10_pruned_40_finetuned

Evaluating wrn28_10_pruned_50_finetuned

Evaluating wrn28_10_pruned_60_finetuned

Evaluating wrn28_10_pruned_70_finetuned


,model,prune_ratio,keep_ratio,acc,nll,ece,ts_ece,ks,ts_ks,mean_sweep_ce,...,ts_nll,acc_std,nll_std,ece_std,ks_std,mean_sweep_ce_std,ts_ece_std,ts_nll_std,ts_ks_std,ts_mean_sweep_ce_std
0,baseline,0.0,1.0,0.9506,0.187703,0.028074,0.015114,0.741059,0.777625,0.028232,...,0.164426,0.002175,0.009528,0.001869,0.071937,0.001853,0.001821,0.007225,0.050897,0.001736
1,wrn28_10_pruned_10_finetuned,0.1,0.9,0.9555,0.177020,0.025458,0.010069,0.321566,0.310601,0.025478,...,0.151273,0.001968,0.009213,0.001751,0.136087,0.001722,0.001586,0.006763,0.135931,0.001519
2,wrn28_10_pruned_20_finetuned,0.2,0.8,0.9533,0.184308,0.027535,0.011372,0.253971,0.233511,0.027518,...,0.156835,0.002349,0.009829,0.001967,0.121222,0.001938,0.001791,0.007283,0.117597,0.001744
3,wrn28_10_pruned_30_finetuned,0.3,0.7,0.9541,0.180211,0.026477,0.012286,0.249786,0.338454,0.026479,...,0.153433,0.002189,0.010122,0.001807,0.187545,0.001787,0.001614,0.007470,0.205464,0.001583
4,wrn28_10_pruned_40_finetuned,0.4,0.6,0.9503,0.192863,0.029823,0.015153,0.718825,0.767549,0.029874,...,0.162506,0.002297,0.009985,0.002036,0.161860,0.002026,0.001830,0.007304,0.212384,0.001791
5,wrn28_10_pruned_50_finetuned,0.5,0.5,0.9517,0.187460,0.027411,0.012184,0.709865,0.248805,0.027738,...,0.159196,0.002094,0.009223,0.001841,0.178376,0.001804,0.001705,0.006768,0.140113,0.001626
6,wrn28_10_pruned_60_finetuned,0.6,0.4,0.9471,0.204641,0.030468,0.015281,0.380032,0.315654,0.030468,...,0.171800,0.002117,0.009955,0.001868,0.011484,0.001837,0.001718,0.007127,0.011107,0.001639
7,wrn28_10_pruned_70_finetuned,0.7,0.3,0.9338,0.239575,0.034285,0.011999,0.744505,0.786403,0.034357,...,0.204122,0.002483,0.010846,0.002158,0.182997,0.002126,0.001910,0.007640,0.234264,0.001796


In [134]:
# --------------------------------------------------
# RAW METRICS TABLE (mean ± std)
# --------------------------------------------------

df_raw = df.copy()

# Accuracy (%)
df_raw["acc (%)"] = (
    (df_raw["acc"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["acc_std"] * 100).round(2).astype(str)
)

# NLL (raw scale)
df_raw["nll"] = (
    df_raw["nll"].round(2).astype(str)
    + " ± "
    + df_raw["nll_std"].round(2).astype(str)
)

# ECE (%)
df_raw["ece (%)"] = (
    (df_raw["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ece_std"] * 100).round(2).astype(str)
)

# KS (%)
df_raw["ks (%)"] = (
    (df_raw["ks"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ks_std"] * 100).round(2).astype(str)
)

# Mean Sweep CE (%)
df_raw["mean_sweep_ce (%)"] = (
    (df_raw["mean_sweep_ce"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["mean_sweep_ce_std"] * 100).round(2).astype(str)
)

df_raw = df_raw[[
    "prune_ratio",
    "acc (%)",
    "nll",
    "ece (%)",
    "ks (%)",
    "mean_sweep_ce (%)"
]]

df_raw.to_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar10_raw_metrics_baseline_GD.csv", index=False)

print("Saved -> cifar_raw_metrics_baseline_GD.csv")
df_raw


Saved -> cifar_raw_metrics_baseline_GD.csv


,prune_ratio,acc (%),nll,ece (%),ks (%),mean_sweep_ce (%)
0,0.0,95.06 ± 0.22,0.19 ± 0.01,2.81 ± 0.19,74.11 ± 7.19,2.82 ± 0.19
1,0.1,95.55 ± 0.2,0.18 ± 0.01,2.55 ± 0.18,32.16 ± 13.61,2.55 ± 0.17
2,0.2,95.33 ± 0.23,0.18 ± 0.01,2.75 ± 0.2,25.4 ± 12.12,2.75 ± 0.19
3,0.3,95.41 ± 0.22,0.18 ± 0.01,2.65 ± 0.18,24.98 ± 18.75,2.65 ± 0.18
4,0.4,95.03 ± 0.23,0.19 ± 0.01,2.98 ± 0.2,71.88 ± 16.19,2.99 ± 0.2
5,0.5,95.17 ± 0.21,0.19 ± 0.01,2.74 ± 0.18,70.99 ± 17.84,2.77 ± 0.18
6,0.6,94.71 ± 0.21,0.2 ± 0.01,3.05 ± 0.19,38.0 ± 1.15,3.05 ± 0.18
7,0.7,93.38 ± 0.25,0.24 ± 0.01,3.43 ± 0.22,74.45 ± 18.3,3.44 ± 0.21


In [135]:
# --------------------------------------------------
# TEMPERATURE SCALED METRICS TABLE (mean ± std)
# --------------------------------------------------

df_ts = df.copy()

# Raw ECE (%)
df_ts["raw_ece (%)"] = (
    (df_ts["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ece_std"] * 100).round(2).astype(str)
)

# Raw NLL
df_ts["raw_nll"] = (
    df_ts["nll"].round(4).astype(str)
    + " ± "
    + df_ts["nll_std"].round(4).astype(str)
)

# TS-ECE (%)
df_ts["ts_ece (%)"] = (
    (df_ts["ts_ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ts_ece_std"] * 100).round(2).astype(str)
)

# TS-NLL
df_ts["ts_nll"] = (
    df_ts["ts_nll"].round(4).astype(str)
    + " ± "
    + (df_ts["ts_nll_std"] * 100).round(4).astype(str)
)

# Temperature (no std typically reported)
df_ts["temperature"] = df_ts["ts_temperature"].round(3)

df_ts = df_ts[[
    "prune_ratio",
    "raw_ece (%)",
    "raw_nll",
    "ts_ece (%)",
    "ts_nll",
    "temperature"
]]

df_ts.to_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar10_temperature_scaled_metrics_baseline_GD.csv", index=False)

print("Saved -> cifar_temperature_scaled_metrics_paper.csv")
df_ts


Saved -> cifar_temperature_scaled_metrics_paper.csv


,prune_ratio,raw_ece (%),raw_nll,ts_ece (%),ts_nll,temperature
0,0.0,2.81 ± 0.19,0.1877 ± 0.0095,1.51 ± 0.18,0.1644 ± 0.7225,1.338
1,0.1,2.55 ± 0.18,0.177 ± 0.0092,1.01 ± 0.16,0.1513 ± 0.6763,1.412
2,0.2,2.75 ± 0.2,0.1843 ± 0.0098,1.14 ± 0.18,0.1568 ± 0.7283,1.415
3,0.3,2.65 ± 0.18,0.1802 ± 0.0101,1.23 ± 0.16,0.1534 ± 0.747,1.408
4,0.4,2.98 ± 0.2,0.1929 ± 0.01,1.52 ± 0.18,0.1625 ± 0.7304,1.404
5,0.5,2.74 ± 0.18,0.1875 ± 0.0092,1.22 ± 0.17,0.1592 ± 0.6768,1.418
6,0.6,3.05 ± 0.19,0.2046 ± 0.01,1.53 ± 0.17,0.1718 ± 0.7127,1.428
7,0.7,3.43 ± 0.22,0.2396 ± 0.0108,1.2 ± 0.19,0.2041 ± 0.764,1.470


In [136]:
@torch.no_grad()
def evaluate_checkpoint_with_bootstrap(
    ckpt_path: str,
    prune_ratio: float,
    val_loader,
    test_loader,
    device,
    n_boot=300,
):
    # -----------------------
    # Load checkpoint
    # -----------------------
    state_dict = torch.load(ckpt_path, map_location=device)

    # --------------------------------------------------
    # CASE 1: Baseline
    # --------------------------------------------------
    if prune_ratio == 0.0:
        model = build_wrn28_10(
            num_classes=10,
            dropout=0.0
        ).to(device)

    # --------------------------------------------------
    # CASE 2: Structured pruned model (WRN-28-10)
    # --------------------------------------------------
    else:

        # 🔥 For extreme pruning (80%, 90%) infer exact widths
        if prune_ratio in [0.8, 0.9]:

            C0 = 16
            C1 = state_dict["layer1.0.conv1.weight"].shape[0]
            C2 = state_dict["layer2.0.conv1.weight"].shape[0]
            C3 = state_dict["layer3.0.conv1.weight"].shape[0]

        # 🔹 For 10–70%, keep proportional reconstruction
        else:
            keep_ratio = 1.0 - prune_ratio

            BASE_C1 = 160
            BASE_C2 = 320
            BASE_C3 = 640

            C0 = 16
            C1 = max(1, int(BASE_C1 * keep_ratio))
            C2 = max(1, int(BASE_C2 * keep_ratio))
            C3 = max(1, int(BASE_C3 * keep_ratio))

        model = WideResNetNoGate(
            stages=[C0, C1, C2, C3],
            depth=28,
            num_classes=10
        ).to(device)

    # -----------------------
    # Load weights
    # -----------------------
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    # -----------------------
    # Collect logits
    # -----------------------
    logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
    logits_test, labels_test = collect_logits_and_labels(model, test_loader, device)

    logits_val_cpu = logits_val.detach().cpu()
    labels_val_cpu = labels_val.detach().cpu()
    logits_test_cpu = logits_test.detach().cpu()
    labels_test_cpu = labels_test.detach().cpu()

    # -----------------------
    # Compute mean metrics
    # -----------------------
    metrics = evaluate_all_metrics(model, val_loader, test_loader, device)

    # -----------------------
    # Bootstrap std
    # -----------------------
    boot = bootstrap_metrics_from_logits(
        logits_val_cpu,
        labels_val_cpu,
        logits_test_cpu,
        labels_test_cpu,
        device,
        n_boot=n_boot
    )

    return {**metrics, **boot}

In [139]:
SEED = 0
BASELINE_CKPT = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/final_model.pth"

SWEEP_DIR = "Baseline_Gate_Decorator/WRN28-10_CIFAR10/FINE_TUNED"


df_rest = evaluate_baseline_and_pruned_models(
    baseline_ckpt=BASELINE_CKPT,
    sweep_dir=SWEEP_DIR,
    prune_ratios=[0.8, 0.9],
    device=DEVICE,
    n_boot=300   
)

df_rest


Evaluating BASELINE

Evaluating wrn28_10_pruned_80_finetuned

Evaluating wrn28_10_pruned_90_finetuned


,model,prune_ratio,keep_ratio,acc,nll,ece,ts_ece,ks,ts_ks,mean_sweep_ce,...,ts_nll,acc_std,nll_std,ece_std,ks_std,mean_sweep_ce_std,ts_ece_std,ts_nll_std,ts_ks_std,ts_mean_sweep_ce_std
0,baseline,0.0,1.0,0.9506,0.187703,0.028074,0.014713,0.741059,0.778608,0.028232,...,0.164160,0.002061,0.008566,0.001738,0.055931,0.001700,0.001663,0.006561,0.048029,0.001593
1,wrn28_10_pruned_80_finetuned,0.8,0.2,0.9148,0.295411,0.038908,0.014698,0.766879,0.780716,0.039000,...,0.261278,0.002624,0.010400,0.002304,0.221511,0.002260,0.002073,0.007558,0.252734,0.001782
2,wrn28_10_pruned_90_finetuned,0.9,0.1,0.8927,0.334507,0.031165,0.011917,0.419482,0.438923,0.031309,...,0.319833,0.002932,0.009462,0.002311,0.194011,0.002322,0.002185,0.008127,0.212958,0.001887


In [140]:
# --------------------------------------------------
# RAW METRICS TABLE (mean ± std)
# --------------------------------------------------

df_raw = df_rest.copy()

# Accuracy (%)
df_raw["acc (%)"] = (
    (df_raw["acc"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["acc_std"] * 100).round(2).astype(str)
)

# NLL (raw scale)
df_raw["nll"] = (
    df_raw["nll"].round(2).astype(str)
    + " ± "
    + df_raw["nll_std"].round(2).astype(str)
)

# ECE (%)
df_raw["ece (%)"] = (
    (df_raw["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ece_std"] * 100).round(2).astype(str)
)

# KS (%)
df_raw["ks (%)"] = (
    (df_raw["ks"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ks_std"] * 100).round(2).astype(str)
)

# Mean Sweep CE (%)
df_raw["mean_sweep_ce (%)"] = (
    (df_raw["mean_sweep_ce"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["mean_sweep_ce_std"] * 100).round(2).astype(str)
)

df_raw = df_raw[[
    "prune_ratio",
    "acc (%)",
    "nll",
    "ece (%)",
    "ks (%)",
    "mean_sweep_ce (%)"
]]

df_raw.to_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar10_raw_metrics_baseline_GD_rest.csv", index=False)

print("Saved -> cifar_raw_metrics_baseline_GD_rest.csv")
df_raw


Saved -> cifar_raw_metrics_baseline_GD_rest.csv


,prune_ratio,acc (%),nll,ece (%),ks (%),mean_sweep_ce (%)
0,0.0,95.06 ± 0.21,0.19 ± 0.01,2.81 ± 0.17,74.11 ± 5.59,2.82 ± 0.17
1,0.8,91.48 ± 0.26,0.3 ± 0.01,3.89 ± 0.23,76.69 ± 22.15,3.9 ± 0.23
2,0.9,89.27 ± 0.29,0.33 ± 0.01,3.12 ± 0.23,41.95 ± 19.4,3.13 ± 0.23


In [143]:
# --------------------------------------------------
# TEMPERATURE SCALED METRICS TABLE (mean ± std)
# --------------------------------------------------

df_ts = df_rest.copy()

# Raw ECE (%)
df_ts["raw_ece (%)"] = (
    (df_ts["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ece_std"] * 100).round(2).astype(str)
)

# Raw NLL
df_ts["raw_nll"] = (
    df_ts["nll"].round(4).astype(str)
    + " ± "
    + df_ts["nll_std"].round(4).astype(str)
)

# TS-ECE (%)
df_ts["ts_ece (%)"] = (
    (df_ts["ts_ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ts_ece_std"] * 100).round(2).astype(str)
)

# TS-NLL
df_ts["ts_nll"] = (
    df_ts["ts_nll"].round(4).astype(str)
    + " ± "
    + (df_ts["ts_nll_std"] * 100).round(4).astype(str)
)

# Temperature (no std typically reported)
df_ts["temperature"] = df_ts["ts_temperature"].round(3)

df_ts = df_ts[[
    "prune_ratio",
    "raw_ece (%)",
    "raw_nll",
    "ts_ece (%)",
    "ts_nll",
    "temperature"
]]

df_ts.to_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar10_temperature_scaled_metrics_baseline_GD_rest.csv", index=False)

print("Saved -> cifar_temperature_scaled_metrics_paper.csv")
df_ts


Saved -> cifar_temperature_scaled_metrics_paper.csv


,prune_ratio,raw_ece (%),raw_nll,ts_ece (%),ts_nll,temperature
0,0.0,2.81 ± 0.17,0.1877 ± 0.0086,1.47 ± 0.17,0.1642 ± 0.6561,1.350
1,0.8,3.89 ± 0.23,0.2954 ± 0.0104,1.47 ± 0.21,0.2613 ± 0.7558,1.381
2,0.9,3.12 ± 0.23,0.3345 ± 0.0095,1.19 ± 0.22,0.3198 ± 0.8127,1.224


In [144]:
import pandas as pd

df_a = pd.read_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar10_raw_metrics_baseline_GD.csv")
df_b = pd.read_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar10_raw_metrics_baseline_GD_rest.csv")



df_merged = pd.concat([df_a, df_b], ignore_index=True)

df_merged = (
    df_merged
    .sort_values("prune_ratio")
    .drop_duplicates(subset="prune_ratio", keep="first")
    .reset_index(drop=True)
)

df_merged.to_csv("Baseline_Gate_Decorator/WRN28-10_CIFAR10/cifar_raw_metrics_baseline_GD_0_90.csv", index=False)

df_merged

,prune_ratio,acc (%),nll,ece (%),ks (%),mean_sweep_ce (%)
0,0.0,95.06 ± 0.22,0.19 ± 0.01,2.81 ± 0.19,74.11 ± 7.19,2.82 ± 0.19
1,0.1,95.55 ± 0.2,0.18 ± 0.01,2.55 ± 0.18,32.16 ± 13.61,2.55 ± 0.17
2,0.2,95.33 ± 0.23,0.18 ± 0.01,2.75 ± 0.2,25.4 ± 12.12,2.75 ± 0.19
3,0.3,95.41 ± 0.22,0.18 ± 0.01,2.65 ± 0.18,24.98 ± 18.75,2.65 ± 0.18
4,0.4,95.03 ± 0.23,0.19 ± 0.01,2.98 ± 0.2,71.88 ± 16.19,2.99 ± 0.2
5,0.5,95.17 ± 0.21,0.19 ± 0.01,2.74 ± 0.18,70.99 ± 17.84,2.77 ± 0.18
6,0.6,94.71 ± 0.21,0.2 ± 0.01,3.05 ± 0.19,38.0 ± 1.15,3.05 ± 0.18
7,0.7,93.38 ± 0.25,0.24 ± 0.01,3.43 ± 0.22,74.45 ± 18.3,3.44 ± 0.21
8,0.8,91.48 ± 0.26,0.3 ± 0.01,3.89 ± 0.23,76.69 ± 22.15,3.9 ± 0.23
9,0.9,89.27 ± 0.29,0.33 ± 0.01,3.12 ± 0.23,41.95 ± 19.4,3.13 ± 0.23
